In [24]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.offsetbox import AnchoredText
import seaborn as sns
from scipy import stats
from scipy.stats import friedmanchisquare
from sklearn.metrics import r2_score
import warnings
import functools
warnings.filterwarnings('ignore')

# =============================================================================
# GLOBAL STYLE SETTINGS
# =============================================================================
def set_modern_style():
    plt.style.use('default')
    mpl.rcParams.update({
        'font.family': 'sans-serif',
        'font.sans-serif': ['Times New Roman', 'DejaVu Sans'],
        'font.size': 11,
        'figure.dpi': 300,
        'axes.labelsize': 13,
        'axes.titlesize': 14,
        'axes.titleweight': 'bold',
        'axes.linewidth': 1.2,
        'axes.grid': True,
        'grid.alpha': 0.15,
        'legend.fontsize': 11,
        'legend.frameon': False,
        'lines.linewidth': 2.5,
        'lines.markersize': 6,
    })
set_modern_style()

# Neon color palette for models (used in per-dataset comparisons)
MODEL_COLORS = {
    'XGBoost': '#f02222',      # red
    'LightGBM': '#89f022',     # lime green
    'NeuralNetwork': '#22f0f0', # cyan
    'Stacking': '#8922f0'       # purple
}

# Distinct colors for datasets (used in per-model across-dataset plots)
SET_COLORS = {
    1: '#f02222',  # vibrant red
    2: '#22f022',  # teal
    3: '#2222f0'   # gold
}

def add_horizontal_legend(ax, ncol=6, frameon=False):
    """Place a horizontal legend above the axes using existing handles."""
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.15),
                  ncol=ncol, frameon=frameon, fontsize=10)
        fig = ax.figure
        fig.subplots_adjust(top=0.85)

def remove_spines(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

def save_plot(fig, name, results_path):
    os.makedirs(results_path, exist_ok=True)
    filepath = os.path.join(results_path, f"{name}.png")
    fig.tight_layout(rect=[0, 0, 1, 0.92])
    fig.savefig(filepath, dpi=300, bbox_inches='tight', facecolor='white', edgecolor='none')
    plt.close(fig)
    print(f"  ✓ Saved: {name}.png")

# =============================================================================
# DATA LOADING
# =============================================================================
base_path = r"C:\Users\Acer\Documents\Machine Learning\CODES FINAL"
model_files = {
    'XGBoost': os.path.join(base_path, "xgboost_visualizer_data.joblib"),
    'LightGBM': os.path.join(base_path, "lightgbm_visualizer_data.joblib"),
    'NeuralNetwork': os.path.join(base_path, "nn_visualizer_data.joblib"),
    'Stacking': os.path.join(base_path, "stacking_visualizer_data.joblib")
}

all_data = {}
for name, path in model_files.items():
    try:
        all_data[name] = joblib.load(path)
        print(f"Loaded {name} with sets: {list(all_data[name].keys())}")
    except Exception as e:
        print(f"Failed to load {name}: {e}")

model_sets = [1, 2, 3]

# =============================================================================
# HELPER FUNCTIONS FOR BOOTSTRAP CACHING
# =============================================================================
@functools.lru_cache(maxsize=128)
def bootstrap_r2_distribution(model_name, set_id, n_bootstrap=200):
    data = all_data[model_name][set_id]
    y_true = data['y_test']
    y_pred = data['y_pred_test']
    n = len(y_true)
    scores = []
    np.random.seed(42)
    for _ in range(n_bootstrap):
        idx = np.random.choice(n, n, replace=True)
        scores.append(r2_score(y_true[idx], y_pred[idx]))
    return np.array(scores)

# =============================================================================
# 1. PER‑DATASET COMPARISON (models side by side)
# =============================================================================
def plot_comparisons_per_dataset(set_id, results_root):
    comp = {}
    for model_name, model_data in all_data.items():
        if set_id in model_data:
            d = model_data[set_id]
            comp[model_name] = {
                'y_test': d['y_test'],
                'y_pred': d['y_pred_test'],
                'residuals': d['residuals_test'],
                'metrics': d['metrics']['test']
            }
    if not comp:
        print(f"No data for ModelSet {set_id}")
        return

    plot_dir = os.path.join(results_root, f"Comparison_Set{set_id}")
    os.makedirs(plot_dir, exist_ok=True)

    # 1. Actual vs Predicted
    fig, ax = plt.subplots(figsize=(10, 8))
    for name, data in comp.items():
        r2 = data['metrics']['R2']
        ax.scatter(data['y_test'], data['y_pred'], alpha=0.7, s=25,
                   color=MODEL_COLORS[name], label=f"{name} (R²={r2:.3f})", edgecolors='white')
    y_min = min(d['y_test'].min() for d in comp.values())
    y_max = max(d['y_test'].max() for d in comp.values())
    ax.plot([y_min, y_max], [y_min, y_max], 'k--', alpha=0.7, label='Perfect')
    ax.set_xlabel('Actual', fontweight='bold')
    ax.set_ylabel('Predicted', fontweight='bold')
    remove_spines(ax)
    add_horizontal_legend(ax, ncol=6)
    save_plot(fig, "01_actual_vs_predicted", plot_dir)

    # 2. Residuals KDE
    fig, ax = plt.subplots(figsize=(10, 6))
    for name, data in comp.items():
        sns.kdeplot(data['residuals'], label=name, color=MODEL_COLORS[name],
                    linewidth=2, ax=ax, fill=False)
    ax.axvline(0, color='red', linestyle='--', linewidth=2, label='Zero error')
    ax.set_xlabel('Residuals', fontweight='bold')
    ax.set_ylabel('Density', fontweight='bold')
    remove_spines(ax)
    add_horizontal_legend(ax, ncol=6)
    save_plot(fig, "02_residuals_kde", plot_dir)

    # 3. Residuals vs Predicted
    fig, ax = plt.subplots(figsize=(10, 6))
    for name, data in comp.items():
        y_pred = data['y_pred']
        resid = data['residuals']
        ax.scatter(y_pred, resid, alpha=0.6, s=15, color=MODEL_COLORS[name], label=name)
        df = pd.DataFrame({'pred': y_pred, 'resid': resid}).sort_values('pred')
        window = max(5, int(len(df) * 0.05))
        ma = df['resid'].rolling(window, center=True).mean()
        ax.plot(df['pred'], ma, '-', color=MODEL_COLORS[name], linewidth=1.5, alpha=0.9)
    ax.axhline(0, color='red', linestyle='--', linewidth=2)
    ax.set_xlabel('Predicted', fontweight='bold')
    ax.set_ylabel('Residuals', fontweight='bold')
    remove_spines(ax)
    add_horizontal_legend(ax, ncol=6)
    save_plot(fig, "03_residuals_vs_predicted", plot_dir)

    # 4. Q-Q plot
    fig, ax = plt.subplots(figsize=(10, 8))
    all_quantiles = []
    for name, data in comp.items():
        resid = data['residuals']
        resid_std = (resid - np.mean(resid)) / np.std(resid)
        osm, osr = stats.probplot(resid_std, dist="norm", fit=False)
        ax.plot(osm, osr, 'o', markersize=4, alpha=0.6, color=MODEL_COLORS[name], label=name)
        all_quantiles.extend(osm)
    min_q = min(all_quantiles); max_q = max(all_quantiles)
    ax.plot([min_q, max_q], [min_q, max_q], 'r--', linewidth=2, label='Normal')
    ax.set_xlabel('Theoretical quantiles', fontweight='bold')
    ax.set_ylabel('Sample quantiles (std residuals)', fontweight='bold')
    remove_spines(ax)
    add_horizontal_legend(ax, ncol=6)
    save_plot(fig, "04_qq_plot", plot_dir)

    # 5. Cumulative absolute error
    fig, ax = plt.subplots(figsize=(10, 6))
    for name, data in comp.items():
        abs_err = np.abs(data['residuals'])
        sorted_err = np.sort(abs_err)
        cum = np.arange(1, len(sorted_err)+1) / len(sorted_err) * 100
        ax.plot(sorted_err, cum, '-', linewidth=1.5, label=name, color=MODEL_COLORS[name])
    ax.set_xlabel('Absolute Error', fontweight='bold')
    ax.set_ylabel('Cumulative Percentage (%)', fontweight='bold')
    remove_spines(ax)
    add_horizontal_legend(ax, ncol=6)
    save_plot(fig, "05_cumulative_error", plot_dir)

    # 6. Metrics bar chart
    metrics_list = ['R2', 'RMSE', 'MAE']
    models = list(comp.keys())
    x = np.arange(len(metrics_list))
    width = 0.2
    fig, ax = plt.subplots(figsize=(12, 6))
    for i, model in enumerate(models):
        values = [comp[model]['metrics'][m] for m in metrics_list]
        offset = (i - len(models)/2) * width + width/2
        bars = ax.bar(x + offset, values, width, label=model,
                      color=MODEL_COLORS[model], edgecolor='white')
        for bar, v in zip(bars, values):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                    f'{v:.3f}', ha='center', va='bottom', fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels(metrics_list)
    ax.set_ylabel('Metric Value', fontweight='bold')
    ax.grid(True, axis='y', alpha=0.1)
    remove_spines(ax)
    add_horizontal_legend(ax, ncol=6)
    save_plot(fig, "06_metrics_bar", plot_dir)

    # 7. Radar chart (legend moved to upper center)
    metrics_radar = ['R2', 'RMSE', 'MAE', 'MAPE']
    raw = {m: [comp[md]['metrics'][m] for md in models] for m in metrics_radar}
    norm = {}
    for m in metrics_radar:
        vals = np.array(raw[m])
        if m == 'R2':
            norm[m] = (vals - vals.min()) / (vals.max() - vals.min() + 1e-9)
        else:
            inv = 1 / (1 + vals)
            norm[m] = (inv - inv.min()) / (inv.max() - inv.min() + 1e-9)
    angles = np.linspace(0, 2*np.pi, len(metrics_radar), endpoint=False).tolist()
    angles += angles[:1]
    fig, ax = plt.subplots(figsize=(8,8), subplot_kw=dict(polar=True))
    for i, md in enumerate(models):
        values = [norm[m][i] for m in metrics_radar] + [norm[metrics_radar[0]][i]]
        ax.plot(angles, values, 'o-', linewidth=2, color=MODEL_COLORS[md], label=md)
        ax.fill(angles, values, alpha=0.1, color=MODEL_COLORS[md])
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metrics_radar, fontsize=10)
    # Place legend in upper center (like other plots)
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.15),
                  ncol=6, frameon=False, fontsize=10)
        fig.subplots_adjust(top=0.85)
    save_plot(fig, "07_radar_chart", plot_dir)

    # 8. Bootstrapped R² KDE
    fig, ax = plt.subplots(figsize=(10, 6))
    for name in comp.keys():
        boot_scores = bootstrap_r2_distribution(name, set_id, n_bootstrap=200)
        sns.kdeplot(boot_scores, label=name, color=MODEL_COLORS[name], linewidth=2, ax=ax, fill=False)
    ax.set_xlabel('R² Score', fontweight='bold')
    ax.set_ylabel('Density', fontweight='bold')
    remove_spines(ax)
    add_horizontal_legend(ax, ncol=6)
    save_plot(fig, "08_bootstrap_kde", plot_dir)

    # 9. Correlation heatmap
    df_pred = pd.DataFrame({name: comp[name]['y_pred'] for name in comp.keys()})
    corr = df_pred.corr()
    fig, ax = plt.subplots(figsize=(8,6))
    sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', center=0,
                square=True, ax=ax, cbar_kws={"label": "Correlation"}, annot_kws={'size': 10})
    remove_spines(ax)
    save_plot(fig, "09_prediction_correlation", plot_dir)

    # 10. Metrics heatmap
    metrics_names = ['R2', 'RMSE', 'MAE', 'MAPE']
    df_metrics = pd.DataFrame(index=models, columns=metrics_names)
    for name in models:
        for m in metrics_names:
            df_metrics.loc[name, m] = comp[name]['metrics'][m]
    fig, ax = plt.subplots(figsize=(8,6))
    sns.heatmap(df_metrics.astype(float), annot=True, fmt='.4f', cmap='viridis', ax=ax,
                cbar_kws={"label": "Value"})
    remove_spines(ax)
    save_plot(fig, "10_metrics_heatmap", plot_dir)

    print(f"Generated 10 comparison plots for Set {set_id}")

# =============================================================================
# 2. PER‑MODEL ACROSS DATASETS COMPARISON (using SET_COLORS)
# =============================================================================
def plot_model_across_datasets(model_name, results_root):
    available = [s for s in model_sets if s in all_data[model_name]]
    if not available:
        return
    plot_dir = os.path.join(results_root, f"{model_name}_AcrossSets")
    os.makedirs(plot_dir, exist_ok=True)

    # (a) Metrics bar chart across sets
    metrics_list = ['R2', 'RMSE', 'MAE']
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for idx, metric in enumerate(metrics_list):
        ax = axes[idx]
        values = [all_data[model_name][s]['metrics']['test'][metric] for s in available]
        bars = ax.bar([f"Set {s}" for s in available], values,
                      color=[SET_COLORS[s] for s in available], alpha=0.8, edgecolor='white')
        for bar, v in zip(bars, values):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{v:.4f}',
                    ha='center', va='bottom')
        ax.set_ylabel(metric, fontweight='bold')
        ax.grid(True, axis='y', alpha=0.1)
        remove_spines(ax)
    save_plot(fig, "metrics_across_sets", plot_dir)

    # (b) Actual vs predicted (three subplots) – each subplot uses its set colour
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for i, s in enumerate(available):
        ax = axes[i]
        d = all_data[model_name][s]
        y_act, y_pred = d['y_test'], d['y_pred_test']
        r2 = d['metrics']['test']['R2']
        ax.scatter(y_act, y_pred, alpha=0.7, s=25, color=SET_COLORS[s], edgecolors='white')
        y_min, y_max = y_act.min(), y_act.max()
        ax.plot([y_min, y_max], [y_min, y_max], 'k--', alpha=0.6)
        ax.set_xlabel('Actual', fontweight='bold')
        ax.set_ylabel('Predicted', fontweight='bold')
        ax.grid(True, alpha=0.1)
        remove_spines(ax)
    save_plot(fig, "actual_vs_pred_across_sets", plot_dir)

    # (c) Residuals KDE across sets – different line colours
    fig, ax = plt.subplots(figsize=(10, 6))
    for s in available:
        resid = all_data[model_name][s]['residuals_test']
        sns.kdeplot(resid, label=f'Set {s}', linewidth=2, ax=ax, color=SET_COLORS[s], fill=False)
    ax.axvline(0, color='red', linestyle='--', linewidth=2, label='Zero error')
    ax.set_xlabel('Residuals', fontweight='bold')
    ax.set_ylabel('Density', fontweight='bold')
    remove_spines(ax)
    add_horizontal_legend(ax, ncol=6)
    save_plot(fig, "residuals_kde_across_sets", plot_dir)

    # (d) Q-Q plot across sets – different colours
    fig, ax = plt.subplots(figsize=(10, 8))
    all_quantiles = []
    for s in available:
        resid = all_data[model_name][s]['residuals_test']
        resid_std = (resid - np.mean(resid)) / np.std(resid)
        osm, osr = stats.probplot(resid_std, dist="norm", fit=False)
        ax.plot(osm, osr, 'o', markersize=4, alpha=0.6, label=f'Set {s}', color=SET_COLORS[s])
        all_quantiles.extend(osm)
    min_q = min(all_quantiles); max_q = max(all_quantiles)
    ax.plot([min_q, max_q], [min_q, max_q], 'r--', linewidth=2, label='Normal')
    ax.set_xlabel('Theoretical quantiles', fontweight='bold')
    ax.set_ylabel('Sample quantiles (std residuals)', fontweight='bold')
    remove_spines(ax)
    add_horizontal_legend(ax, ncol=6)
    save_plot(fig, "qq_plot_across_sets", plot_dir)

    # (e) Cumulative error across sets – different colours
    fig, ax = plt.subplots(figsize=(10, 6))
    for s in available:
        abs_err = np.abs(all_data[model_name][s]['residuals_test'])
        sorted_err = np.sort(abs_err)
        cum = np.arange(1, len(sorted_err)+1) / len(sorted_err) * 100
        ax.plot(sorted_err, cum, '-', linewidth=2, label=f'Set {s}', color=SET_COLORS[s])
    ax.set_xlabel('Absolute Error', fontweight='bold')
    ax.set_ylabel('Cumulative Percentage (%)', fontweight='bold')
    remove_spines(ax)
    add_horizontal_legend(ax, ncol=6)
    save_plot(fig, "cumulative_error_across_sets", plot_dir)

    print(f"Generated across‑sets comparison plots for {model_name}")

# =============================================================================
# 3. ADVANCED CROSS‑MODEL PLOTS (unchanged except legend ncol)
# =============================================================================
def taylor_diagram(comp_data, set_id, ax=None):
    import math
    if ax is None:
        fig, ax = plt.subplots(figsize=(8,8), subplot_kw=dict(polar=True))
    else:
        fig = ax.figure
    ref_std = np.std(list(comp_data.values())[0]['y_test'])
    models = []
    r_list = []
    std_ratio_list = []
    for name, data in comp_data.items():
        y_act = data['y_test']
        y_pred = data['y_pred']
        r = np.corrcoef(y_act, y_pred)[0,1]
        std_pred = np.std(y_pred)
        std_ratio = std_pred / ref_std
        models.append(name)
        r_list.append(r)
        std_ratio_list.append(std_ratio)
    theta = [math.acos(min(0.9999, r)) for r in r_list]
    radii = std_ratio_list
    for i, name in enumerate(models):
        ax.plot(theta[i], radii[i], 'o', markersize=8, color=MODEL_COLORS[name], label=name)
    ax.plot(0, 1, 'ko', markersize=10, label='Reference')
    ax.set_theta_zero_location('E')
    ax.set_theta_direction(-1)
    ax.set_ylim(0, 1.5)
    ax.legend(loc='upper right', bbox_to_anchor=(1.2,1.0), frameon=False, ncol=6)
    return fig

def bland_altman_plot(model1, model2, data1, data2, set_id, results_path):
    y1 = data1['y_pred']
    y2 = data2['y_pred']
    mean = (y1 + y2) / 2
    diff = y1 - y2
    mean_diff = np.mean(diff)
    std_diff = np.std(diff)
    loa_upper = mean_diff + 1.96 * std_diff
    loa_lower = mean_diff - 1.96 * std_diff
    fig, ax = plt.subplots(figsize=(8,6))
    ax.scatter(mean, diff, alpha=0.5, s=20, color='gray', edgecolors='white')
    ax.axhline(mean_diff, color='red', linestyle='-', linewidth=2, label=f'Mean diff: {mean_diff:.4f}')
    ax.axhline(loa_upper, color='orange', linestyle='--', linewidth=1.5, label=f'+1.96 SD: {loa_upper:.4f}')
    ax.axhline(loa_lower, color='orange', linestyle='--', linewidth=1.5, label=f'-1.96 SD: {loa_lower:.4f}')
    ax.set_xlabel('Average Prediction', fontweight='bold')
    ax.set_ylabel('Difference (Model1 - Model2)', fontweight='bold')
    ax.grid(True, alpha=0.1)
    remove_spines(ax)
    add_horizontal_legend(ax, ncol=6)
    name = f"bland_altman_{model1}_vs_{model2}_set{set_id}"
    save_plot(fig, name, results_path)

def model_ranking_plot(all_data, results_root):
    ranks = {model: [] for model in all_data.keys()}
    metrics = ['R2', 'RMSE', 'MAE', 'MAPE']
    for set_id in model_sets:
        for metric in metrics:
            vals = {}
            for model, data_dict in all_data.items():
                if set_id in data_dict:
                    vals[model] = data_dict[set_id]['metrics']['test'][metric]
            if vals:
                if metric == 'R2':
                    sorted_models = sorted(vals, key=vals.get, reverse=True)
                else:
                    sorted_models = sorted(vals, key=vals.get)
                rank = 1
                for m in sorted_models:
                    ranks[m].append(rank)
                    rank += 1
    avg_rank = {m: np.mean(ranks[m]) for m in ranks}
    models_sorted = sorted(avg_rank, key=avg_rank.get)
    fig, ax = plt.subplots(figsize=(8,6))
    bars = ax.bar(models_sorted, [avg_rank[m] for m in models_sorted],
                  color=[MODEL_COLORS[m] for m in models_sorted])
    ax.set_ylabel('Average Rank (lower is better)', fontweight='bold')
    for bar, m in zip(bars, models_sorted):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                f'{avg_rank[m]:.2f}', ha='center', va='bottom')
    ax.grid(True, alpha=0.1, axis='y')
    remove_spines(ax)
    save_plot(fig, "model_ranking", results_root)

def friedman_nemenyi(all_data, results_root):
    r2_values = {}
    for model in all_data.keys():
        r2_values[model] = []
        for set_id in model_sets:
            if set_id in all_data[model]:
                r2_values[model].append(all_data[model][set_id]['metrics']['test']['R2'])
    try:
        stat, p = friedmanchisquare(*[r2_values[m] for m in all_data.keys()])
        print(f"Friedman test: statistic={stat:.4f}, p={p:.4f}")
    except Exception as e:
        print(f"Friedman test failed: {e}")
        return
    ranks = {}
    all_models = list(all_data.keys())
    for model in all_models:
        rank_sum = 0
        n = 0
        for set_id in model_sets:
            vals_set = []
            for m in all_models:
                if set_id in all_data[m]:
                    vals_set.append(all_data[m][set_id]['metrics']['test']['R2'])
                else:
                    vals_set.append(np.nan)
            valid = ~np.isnan(vals_set)
            if np.sum(valid) >= 2:
                subset_vals = np.array(vals_set)[valid]
                subset_models = np.array(all_models)[valid]
                order = np.argsort(subset_vals)[::-1]
                for rank, idx in enumerate(order, start=1):
                    if subset_models[idx] == model:
                        rank_sum += rank
                        n += 1
        if n > 0:
            ranks[model] = rank_sum / n
    k = len(all_models)
    N = len(model_sets)
    q_alpha = 2.728
    CD = q_alpha * np.sqrt(k*(k+1)/(6*N))
    models_sorted = sorted(ranks, key=ranks.get)
    positions = np.arange(len(models_sorted))
    fig, ax = plt.subplots(figsize=(10,6))
    ax.scatter(positions, [ranks[m] for m in models_sorted], s=100, color='black', zorder=5)
    for i in range(len(models_sorted)):
        for j in range(i+1, len(models_sorted)):
            if abs(ranks[models_sorted[i]] - ranks[models_sorted[j]]) < CD:
                ax.plot([positions[i], positions[j]], [ranks[models_sorted[i]], ranks[models_sorted[j]]],
                        'k-', alpha=0.3)
    ax.set_xticks(positions)
    ax.set_xticklabels(models_sorted, rotation=45, ha='right')
    ax.set_ylabel('Average Rank', fontweight='bold')
    ax.grid(True, alpha=0.1, axis='y')
    remove_spines(ax)
    save_plot(fig, "critical_difference_diagram", results_root)

def grouped_bar_all_models_sets(all_data, results_root):
    metrics = ['R2', 'RMSE', 'MAE', 'MAPE']
    for metric in metrics:
        fig, ax = plt.subplots(figsize=(12,6))
        x = np.arange(len(model_sets))
        width = 0.2
        for i, model in enumerate(all_data.keys()):
            values = []
            for set_id in model_sets:
                if set_id in all_data[model]:
                    values.append(all_data[model][set_id]['metrics']['test'][metric])
                else:
                    values.append(np.nan)
            offset = (i - len(all_data)/2) * width + width/2
            bars = ax.bar(x + offset, values, width, label=model,
                          color=MODEL_COLORS[model], edgecolor='white')
            for bar, v in zip(bars, values):
                if not np.isnan(v):
                    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                            f'{v:.4f}', ha='center', va='bottom', fontsize=8)
        ax.set_xticks(x)
        ax.set_xticklabels([f'Set {s}' for s in model_sets])
        ax.set_ylabel(metric, fontweight='bold')
        ax.legend(frameon=False, ncol=6, loc='upper center', bbox_to_anchor=(0.5, 1.15))
        fig.subplots_adjust(top=0.85)
        ax.grid(True, alpha=0.1, axis='y')
        remove_spines(ax)
        save_plot(fig, f"grouped_bar_{metric}", results_root)

def absolute_residual_boxplots(all_data, results_root):
    fig, ax = plt.subplots(figsize=(12,6))
    data_to_plot = []
    labels = []
    model_list = []
    for model in all_data.keys():
        for set_id in model_sets:
            if set_id in all_data[model]:
                abs_resid = np.abs(all_data[model][set_id]['residuals_test'])
                data_to_plot.append(abs_resid)
                labels.append(f"{model}\nSet{set_id}")
                model_list.append(model)
    bp = ax.boxplot(data_to_plot, labels=labels, patch_artist=True,
                    boxprops=dict(linewidth=1.5), whiskerprops=dict(linewidth=1.5),
                    capprops=dict(linewidth=1.5), medianprops=dict(color='red', linewidth=2))
    for patch, model in zip(bp['boxes'], model_list):
        patch.set_facecolor(MODEL_COLORS[model])
        patch.set_alpha(0.7)
    ax.set_ylabel('Absolute Residuals', fontweight='bold')
    ax.grid(True, alpha=0.1, axis='y')
    remove_spines(ax)
    save_plot(fig, "absolute_residuals_boxplots", results_root)

def improvement_heatmap(all_data, results_root):
    models = list(all_data.keys())
    n = len(models)
    improvement = np.zeros((n, n))
    for i, m1 in enumerate(models):
        for j, m2 in enumerate(models):
            if i == j:
                improvement[i,j] = 0
            else:
                diffs = []
                for set_id in model_sets:
                    if set_id in all_data[m1] and set_id in all_data[m2]:
                        r2_1 = all_data[m1][set_id]['metrics']['test']['R2']
                        r2_2 = all_data[m2][set_id]['metrics']['test']['R2']
                        diffs.append(r2_1 - r2_2)
                improvement[i,j] = np.mean(diffs) if diffs else np.nan
    fig, ax = plt.subplots(figsize=(8,6))
    sns.heatmap(improvement, annot=True, fmt='.4f', cmap='RdBu_r', center=0,
                xticklabels=models, yticklabels=models, ax=ax,
                cbar_kws={'label': 'ΔR² (row - column)'})
    remove_spines(ax)
    save_plot(fig, "improvement_heatmap", results_root)

def violin_errors(all_data, results_root):
    df_list = []
    for model in all_data.keys():
        for set_id in model_sets:
            if set_id in all_data[model]:
                resid = all_data[model][set_id]['residuals_test']
                for r in resid:
                    df_list.append({'Model': model, 'Set': f'Set{set_id}', 'Residual': r})
    df = pd.DataFrame(df_list)
    set_palette = {f'Set{s}': SET_COLORS[s] for s in model_sets}
    fig, ax = plt.subplots(figsize=(12,6))
    sns.violinplot(x='Model', y='Residual', hue='Set', data=df,
                   palette=set_palette, split=False, ax=ax)
    ax.axhline(0, color='red', linestyle='--', linewidth=2)
    ax.set_ylabel('Residual', fontweight='bold')
    ax.grid(True, alpha=0.1, axis='y')
    remove_spines(ax)
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.15),
                  ncol=6, frameon=False)
        fig.subplots_adjust(top=0.85)
    save_plot(fig, "violin_errors", results_root)

def cumulative_lift(all_data, results_root):
    for set_id in model_sets:
        fig, ax = plt.subplots(figsize=(10,6))
        for model, data_dict in all_data.items():
            if set_id in data_dict:
                d = data_dict[set_id]
                y_act = d['y_test']
                y_pred = d['y_pred_test']
                sorted_idx = np.argsort(y_pred)
                cum_actual = np.cumsum(y_act[sorted_idx])
                total = np.sum(y_act)
                if total == 0:
                    cum_actual = np.arange(1, len(cum_actual)+1) / len(cum_actual)
                else:
                    cum_actual = cum_actual / total
                x_axis = np.arange(1, len(cum_actual)+1) / len(cum_actual)
                ax.plot(x_axis, cum_actual, label=model, color=MODEL_COLORS[model], linewidth=2, alpha=0.7)
        ax.plot([0,1], [0,1], 'k--', alpha=0.7, label='Random')
        ax.set_xlabel('Proportion of samples (sorted by prediction)', fontweight='bold')
        ax.set_ylabel('Cumulative proportion of actual', fontweight='bold')
        remove_spines(ax)
        add_horizontal_legend(ax, ncol=6)
        save_plot(fig, f"lift_curve_set{set_id}", results_root)

def residual_std_comparison(all_data, results_root):
    fig, ax = plt.subplots(figsize=(10,6))
    x = np.arange(len(model_sets))
    width = 0.2
    for i, model in enumerate(all_data.keys()):
        stds = []
        for set_id in model_sets:
            if set_id in all_data[model]:
                stds.append(np.std(all_data[model][set_id]['residuals_test']))
            else:
                stds.append(np.nan)
        offset = (i - len(all_data)/2) * width + width/2
        bars = ax.bar(x + offset, stds, width, label=model,
                      color=MODEL_COLORS[model], edgecolor='white')
        for bar, v in zip(bars, stds):
            if not np.isnan(v):
                ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                        f'{v:.4f}', ha='center', va='bottom', fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels([f'Set {s}' for s in model_sets])
    ax.set_ylabel('Standard Deviation of Residuals', fontweight='bold')
    ax.grid(True, alpha=0.1, axis='y')
    remove_spines(ax)
    add_horizontal_legend(ax, ncol=6)
    save_plot(fig, "residual_std_comparison", results_root)

# =============================================================================
# MAIN EXECUTION
# =============================================================================
def main():
    results_root = "./comparison_plots"
    print("="*70)
    print("Generating cross‑model comparison plots for all datasets")
    print("="*70)

    for set_id in model_sets:
        print(f"\n--- Comparison for ModelSet {set_id} ---")
        plot_comparisons_per_dataset(set_id, results_root)

    for model_name in all_data.keys():
        print(f"\n--- Comparison for {model_name} across datasets ---")
        plot_model_across_datasets(model_name, results_root)

    print("\n✅ All basic comparison plots saved in folder:", results_root)

    print("\n--- Generating advanced comparison plots ---")
    for set_id in model_sets:
        comp = {}
        for model_name, model_data in all_data.items():
            if set_id in model_data:
                d = model_data[set_id]
                comp[model_name] = {'y_test': d['y_test'], 'y_pred': d['y_pred_test']}
        if comp:
            fig = taylor_diagram(comp, set_id)
            save_plot(fig, f"taylor_diagram_set{set_id}", results_root)

    for set_id in model_sets:
        comp = {}
        for model_name, model_data in all_data.items():
            if set_id in model_data:
                d = model_data[set_id]
                comp[model_name] = {'y_pred': d['y_pred_test'], 'y_test': d['y_test']}
        model_names = list(comp.keys())
        for i in range(len(model_names)):
            for j in range(i+1, len(model_names)):
                bland_altman_plot(model_names[i], model_names[j], comp[model_names[i]], comp[model_names[j]], set_id, results_root)

    model_ranking_plot(all_data, results_root)
    friedman_nemenyi(all_data, results_root)
    grouped_bar_all_models_sets(all_data, results_root)
    absolute_residual_boxplots(all_data, results_root)
    improvement_heatmap(all_data, results_root)
    violin_errors(all_data, results_root)
    cumulative_lift(all_data, results_root)
    residual_std_comparison(all_data, results_root)

    print("\n✅ All advanced plots generated.")

if __name__ == "__main__":
    main()

Loaded XGBoost with sets: [1, 2, 3]
Loaded LightGBM with sets: [1, 2, 3]
Loaded NeuralNetwork with sets: [1, 2, 3]
Loaded Stacking with sets: [1, 2, 3]
Generating cross‑model comparison plots for all datasets

--- Comparison for ModelSet 1 ---
  ✓ Saved: 01_actual_vs_predicted.png
  ✓ Saved: 02_residuals_kde.png
  ✓ Saved: 03_residuals_vs_predicted.png
  ✓ Saved: 04_qq_plot.png
  ✓ Saved: 05_cumulative_error.png
  ✓ Saved: 06_metrics_bar.png
  ✓ Saved: 07_radar_chart.png
  ✓ Saved: 08_bootstrap_kde.png
  ✓ Saved: 09_prediction_correlation.png
  ✓ Saved: 10_metrics_heatmap.png
Generated 10 comparison plots for Set 1

--- Comparison for ModelSet 2 ---
  ✓ Saved: 01_actual_vs_predicted.png
  ✓ Saved: 02_residuals_kde.png
  ✓ Saved: 03_residuals_vs_predicted.png
  ✓ Saved: 04_qq_plot.png
  ✓ Saved: 05_cumulative_error.png
  ✓ Saved: 06_metrics_bar.png
  ✓ Saved: 07_radar_chart.png
  ✓ Saved: 08_bootstrap_kde.png
  ✓ Saved: 09_prediction_correlation.png
  ✓ Saved: 10_metrics_heatmap.png
Gen